# Hotel Search Ranking, Part 1, the Data and Position Bias

**The question behind this project:** when someone searches for a hotel, which hotels should go at the top of the list so that they actually book?

This notebook does two things before any modelling:

1. Get the Expedia data into a manageable sample and look at it properly.
2. Measure **position bias**: how much of a hotel's clicks come simply from being high on the page, rather than from the hotel itself.

The second one matters because every click in this data was influenced by where Expedia put the hotel. If I train a model on clicks without understanding that, the model partly learns "Expedia ranked this high", not "people want this hotel".

Luckily, Expedia showed about 30% of searches in **random order**. In those searches a hotel's position has nothing to do with how good it is, so they let me separate the effect of position from the effect of the hotel.

Dataset: [Personalize Expedia Hotel Searches, ICDM 2013 (Kaggle)](https://www.kaggle.com/c/expedia-personalized-sort)

Author: Siddharth Suman

## Setup

I run this in Google Colab. The full training file is about 10 million rows, so the first run downloads it from Kaggle, keeps a 25% sample of searches, and saves that sample to my Google Drive. Every later run (and the later notebooks) just load the saved sample, which takes seconds.

**Before the first run:** add your Kaggle username and API key as Colab secrets called `KAGGLE_USERNAME` and `KAGGLE_KEY` (key icon in the left sidebar), and accept the competition rules on the Kaggle page. The README explains both steps.

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 60)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK_DIR = '/content/drive/MyDrive/expedia-hotel-ranking'
    IN_COLAB = True
except ImportError:
    WORK_DIR = 'data'
    IN_COLAB = False

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs('images', exist_ok=True)
SAMPLE_PATH = os.path.join(WORK_DIR, 'train_sample.parquet')
print("Sample file:", SAMPLE_PATH, "| already exists:", os.path.exists(SAMPLE_PATH))

In [ ]:
# Download from Kaggle, only if I don't already have the sample saved.
def get_train_csv():
    if not IN_COLAB:
        return 'data/train.csv'   # running locally: put the Kaggle file here yourself

    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

    import kagglehub
    path = kagglehub.competition_download('expedia-personalized-sort')

    # The download is sometimes a folder, sometimes a zip inside a folder
    found = glob.glob(os.path.join(path, '**', 'train.csv'), recursive=True)
    if not found:
        import zipfile
        for z in glob.glob(os.path.join(path, '**', '*.zip'), recursive=True):
            zipfile.ZipFile(z).extractall(path)
        found = glob.glob(os.path.join(path, '**', 'train.csv'), recursive=True)
    return found[0]

if not os.path.exists(SAMPLE_PATH):
    train_csv = get_train_csv()
    print("Reading", train_csv)

    # 10M rows is a lot for free Colab. I keep every 4th search (srch_id divisible by 4).
    # Sampling whole searches, not random rows, matters: a ranking only makes
    # sense if I have every hotel that was shown in that search.
    # Missing values in this file are written as the text "NULL".
    parts = []
    for chunk in pd.read_csv(train_csv, na_values=['NULL'], chunksize=1_000_000):
        parts.append(chunk[chunk['srch_id'] % 4 == 0])
    df = pd.concat(parts, ignore_index=True)

    # Smaller number types to save memory
    for col in df.select_dtypes('float64').columns:
        df[col] = df[col].astype('float32')
    for col in df.select_dtypes('int64').columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')

    df.to_parquet(SAMPLE_PATH, index=False)
    print("Saved sample to Drive")

df = pd.read_parquet(SAMPLE_PATH)
print(f"{len(df):,} rows, {df['srch_id'].nunique():,} searches, {df['prop_id'].nunique():,} hotels")
df.head()

## First look

Each row is one hotel shown in one search. The columns I care about most right now:

- `srch_id`: which search the row belongs to
- `position`: where the hotel appeared on the results page (1 = top)
- `random_bool`: 1 if this search was shown in random order
- `click_bool`, `booking_bool`: did the user click this hotel, did they book it
- `price_usd`, `prop_starrating`, `prop_review_score`, `prop_location_score2`: what the user could see or what describes the hotel

In [ ]:
searches = df.groupby('srch_id').agg(
    hotels_shown=('prop_id', 'size'),
    clicks=('click_bool', 'sum'),
    booked=('booking_bool', 'max'),
    random_order=('random_bool', 'first'),
)

print(f"Hotels shown per search:   median {searches['hotels_shown'].median():.0f}, max {searches['hotels_shown'].max()}")
print(f"Click rate per hotel shown: {df['click_bool'].mean():.2%}")
print(f"Booking rate per hotel shown: {df['booking_bool'].mean():.2%}")
print(f"Searches ending in a booking: {searches['booked'].mean():.1%}")
print(f"Searches shown in random order: {searches['random_order'].mean():.1%}")

print("\nSearches ending in a booking, by how the list was ordered:")
print(searches.groupby('random_order')['booked'].mean().rename({0: 'Expedia order', 1: 'Random order'}).map('{:.1%}'.format))

✏️ **My notes (write 2 to 4 sentences in your own words):**
- How rare are clicks and bookings? What does that mean for a model (most rows are "nothing happened")?
- Compare the booking rate for searches in Expedia's order with searches in random order. What does that gap tell you about how much a good ranking is worth?

## Data quirks to know before modelling

In [ ]:
key_cols = ['price_usd', 'prop_starrating', 'prop_review_score', 'prop_location_score1',
            'prop_location_score2', 'prop_log_historical_price', 'orig_destination_distance',
            'visitor_hist_starrating', 'visitor_hist_adr_usd', 'srch_query_affinity_score']
missing = df[key_cols].isna().mean().sort_values(ascending=False)
print("Share missing:")
print(missing.map('{:.0%}'.format))

print("\nprice_usd percentiles:")
print(df['price_usd'].quantile([0.01, 0.25, 0.5, 0.75, 0.99, 1.0]).round(0))

# Star rating 0 and review score 0 are not real zeros: they mean "no rating"
print(f"\nHotels with star rating 0: {(df['prop_starrating'] == 0).mean():.1%}")
print(f"Hotels with review score 0: {(df['prop_review_score'] == 0).mean():.1%}")

✏️ **My notes:**
- Which columns are mostly missing? Are any of them still worth using later?
- Look at the top price percentiles. Expedia's data notes say `price_usd` can be per night or for the whole stay depending on the country, and include different taxes. What would you do about that before comparing prices?

## Position bias: how much does being near the top matter?

For each position on the page I calculate the click rate, separately for searches in Expedia's own order and searches in random order.

- In **Expedia order**, the top hotels get more clicks for two reasons: they're at the top, *and* Expedia put good hotels there.
- In **random order**, hotels are placed by chance. Any drop in clicks as you go down the page there is the effect of position alone.

In [ ]:
df['ordering'] = np.where(df['random_bool'] == 1, 'Random order', 'Expedia order')

by_pos = (df.groupby(['ordering', 'position'])
            .agg(click_rate=('click_bool', 'mean'),
                 booking_rate=('booking_bool', 'mean'),
                 rows=('click_bool', 'size'))
            .reset_index())

# Positions far down the page have few rows, so I stop at 35
by_pos = by_pos[by_pos['position'] <= 35]

colors = {'Expedia order': '#1F4E79', 'Random order': '#C44B4B'}
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Click and booking rate by position on the page', fontsize=14, fontweight='bold')

for name, grp in by_pos.groupby('ordering'):
    axes[0].plot(grp['position'], grp['click_rate'] * 100, marker='o', markersize=3,
                 color=colors[name], label=name)
    axes[1].plot(grp['position'], grp['booking_rate'] * 100, marker='o', markersize=3,
                 color=colors[name], label=name)

axes[0].set_title('Click rate')
axes[1].set_title('Booking rate')
for ax in axes:
    ax.set_xlabel('Position on page (1 = top)')
    ax.set_ylabel('% of hotels shown')
    ax.legend()

plt.tight_layout()
plt.savefig('images/01_position_bias.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Put numbers on it, using searches shown in random order only (position effect alone)
rnd = df[df['random_bool'] == 1]
top = rnd[rnd['position'] <= 5]['click_bool'].mean()
mid = rnd[rnd['position'].between(11, 25)]['click_bool'].mean()
print(f"Random order: click rate at positions 1 to 5:   {top:.2%}")
print(f"Random order: click rate at positions 11 to 25: {mid:.2%}")
print(f"Being in the top 5 instead of the middle: {top / mid:.1f}x the clicks, from position alone")

exp = df[df['random_bool'] == 0]
exp_top = exp[exp['position'] <= 5]['click_bool'].mean()
print(f"\nExpedia order: click rate at positions 1 to 5: {exp_top:.2%}")
print(f"Top 5 in Expedia order vs top 5 in random order: {exp_top / top:.1f}x")
print("(that extra part is Expedia's ranking putting better hotels on top)")

✏️ **My notes:**
- In random order, how much does the click rate fall from the top to the middle of the page? That's pure position effect.
- How much higher is the top of the page in Expedia order than in random order? That's roughly what a good ranking adds on top of position.
- Do you see any odd jumps at particular positions? (Page layout can do this. Note them, but don't overthink them.)

## Clicks vs bookings: does position fool people into clicking?

If position pushes people to click hotels they don't really want, then clicks from the top of the page should turn into bookings *less* often. I check this by looking only at clicked hotels in searches shown in random order and asking: what share of clicks became a booking, at each part of the page?

In [ ]:
clicked = rnd[rnd['click_bool'] == 1].copy()
clicked['position_group'] = pd.cut(clicked['position'], bins=[0, 5, 10, 15, 20, 25, 40],
                                   labels=['1 to 5', '6 to 10', '11 to 15', '16 to 20', '21 to 25', '26+'])

conversion = (clicked.groupby('position_group', observed=True)
                     .agg(clicks=('click_bool', 'size'),
                          booked_after_click=('booking_bool', 'mean')))
print(conversion.assign(booked_after_click=conversion['booked_after_click'].map('{:.0%}'.format)))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(conversion.index.astype(str), conversion['booked_after_click'] * 100, color='#2874A6')
ax.set_title('Random order: share of clicks that became a booking', fontweight='bold')
ax.set_xlabel('Position on page')
ax.set_ylabel('% of clicks booked')
plt.tight_layout()
plt.savefig('images/01_booking_after_click.png', dpi=150, bbox_inches='tight')
plt.show()

✏️ **My notes:**
- Do clicks near the top turn into bookings less often than clicks further down? If so, clicks at the top of the page are partly "free" clicks from position, and a click is weaker evidence than a booking.
- What does this mean for training a ranking model? (Hint: the competition gave a booking 5 points and a click 1 point. Does that weighting make sense given what you found?)

## What I'm taking into the next notebook

✏️ Write 3 bullet points: the main things you learned here that should shape the model (for example: don't use `position` as a feature, since it won't exist for a new search; treat bookings as much stronger signal than clicks; how to handle price).